# Halden Energy — Quarter 12: what should Halden become?
**Python notebook (the same data as the workbook)**

The board gives you $1,800M over five years; $600M keeps what Halden has running; $1,200M is yours to place. Five things are on the table, and each is worth money in some futures and loses it in others. There is no set that wins everywhere.

Unzip the data files next to this notebook before you run it.

In [ ]:
import itertools
import pandas as pd

envelope = pd.read_csv('data/the_envelope.csv').set_index('parameter')['value']
projects = pd.read_csv('data/the_projects.csv').set_index('project')
groups = pd.read_csv('data/the_groups.csv').set_index('group')
futures = pd.read_csv('data/the_futures.csv')
carbons = futures[futures.kind == 'carbon'].set_index('future')['value']
demands = futures[futures.kind == 'demand'].set_index('future')['value']
print(projects[['group', 'cost_musd']])

## Part 1 — every project in every future

Value = value at $40 carbon and slow decline + change per dollar of carbon × (carbon − 40) + change per demand step × step.

In [ ]:
def value(project, carbon, step):
    p = projects.loc[project]
    return p['value_at_40_carbon_slow_decline'] + p['change_per_dollar_of_carbon'] * (carbon - 40) + p['change_per_step_of_demand_decline'] * step

grid = pd.DataFrame({f'{c} / {d}': {proj: value(proj, cv, dv) for proj in projects.index} for c, cv in carbons.items() for d, dv in demands.items()})
print(grid.round(0))
print()
print('worst and best future for each project:')
print(pd.DataFrame({'worst': grid.min(axis=1), 'best': grid.max(axis=1)}).round(0))

## Part 2 — what fits

The envelope after the floor, widened by the sale if you make it; the board's cap on each group.

In [ ]:
free = envelope['free_to_place_musd']
sale = envelope['sale_of_european_stations_musd']

def fits(chosen):
    cost = sum(projects.loc[p, 'cost_musd'] for p in chosen)   # the sale has a negative cost: it adds room
    if cost > free:
        return False
    for g, row in groups.iterrows():
        spend = sum(projects.loc[p, 'cost_musd'] for p in chosen if projects.loc[p, 'group'] == g and projects.loc[p, 'cost_musd'] > 0)
        if spend > row['cap_musd']:
            return False
    return True

sets = [c for n in range(1, 6) for c in itertools.combinations(projects.index, n) if fits(c)]
print(f'{len(sets)} sets of projects fit')
for s in sets:
    print('  ' + ' + '.join(s))
print()
print('Helix at Rotterdam + offshore wind fits:', fits(('Helix at Rotterdam', 'Offshore wind off Norway')))
print('...with the sale:', fits(('Helix at Rotterdam', 'Offshore wind off Norway', 'Sell the European stations')))

## Part 3 — a set across the futures

In [ ]:
def set_value(chosen):
    return pd.Series({col: sum(grid.loc[p, col] for p in chosen) for col in grid.columns})

for chosen in [('Expand in the Permian',), ('Convert part of Rotterdam to biofuels', 'Offshore wind off Norway'), ('Helix at Rotterdam', 'Offshore wind off Norway', 'Sell the European stations')]:
    v = set_value(chosen)
    print(' + '.join(chosen))
    print(f'   worst {v.min():.0f}  best {v.max():.0f}   (worst in: {v.idxmin()}; best in: {v.idxmax()})')

## Your analysis

Fill in the `# TODO` cells.

In [ ]:
# TODO: your set of projects. Does it fit? What does it cost over five years, and per quarter?
# TODO: its value in its worst future and its best. What would you have to believe for it to be right?
# TODO: what does Halden become in five years under your set? Is it the sentence you wrote on your first day?

## Your decision

- Which projects go ahead, and do you sell the European stations?
- Which future are you betting on, and what happens to you in the others?
- What will you say to the board in January?